# 03 - Silver Validation

This notebook validates Bronze data, writes clean records to Silver, and writes invalid records to a rejected-record layer.

In [ ]:
%%configure -f
{
  "conf": {
    "spark.sql.extensions": "io.delta.sql.DeltaSparkSessionExtension",
    "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog"
  }
}

In [ ]:
from datetime import datetime
from pyspark.sql.functions import col, lit, trim, when

BRONZE_PATH = "s3://loanshield-bronze/"
SILVER_PATH = "s3://loanshield-silver/"
REJECTED_PATH = "s3://loanshield-rejected/"

REQUIRED_COLUMNS = [
    "id", "loan_amnt", "term", "int_rate", "grade", "emp_length",
    "home_ownership", "annual_inc", "loan_status", "dti", "addr_state", "fico_range_low"
]
VALID_LOAN_STATUS = ["Fully Paid", "Current", "Charged Off", "Late (31-120 days)", "In Grace Period", "Late (16-30 days)", "Default"]
VALID_GRADES = ["A", "B", "C", "D", "E", "F", "G"]

In [ ]:
start_time = datetime.now()
df_bronze = spark.read.format("delta").load(BRONZE_PATH)
missing_columns = [c for c in REQUIRED_COLUMNS if c not in df_bronze.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df_selected = df_bronze.select(REQUIRED_COLUMNS)
df_clean = df_selected.filter(~col("id").cast("string").rlike("[a-zA-Z]"))
df_clean = df_clean.filter(~col("dti").cast("string").rlike("[a-zA-Z]") & ~col("fico_range_low").cast("string").rlike("[a-zA-Z]"))

df_typed = (
    df_clean
    .withColumn("loan_amnt", col("loan_amnt").cast("double"))
    .withColumn("annual_inc", col("annual_inc").cast("double"))
    .withColumn("dti", col("dti").cast("double"))
    .withColumn("fico_range_low", col("fico_range_low").cast("double"))
    .withColumn("grade", trim(col("grade")))
    .withColumn("loan_status", trim(col("loan_status")))
    .withColumn("term", trim(col("term")))
)

df_validated = df_typed.withColumn(
    "rejection_reason",
    when(col("annual_inc").isNull(), lit("NULL_ANNUAL_INC"))
    .when(col("annual_inc") <= 0, lit("ZERO_ANNUAL_INC"))
    .when(col("annual_inc") > 1000000, lit("EXTREME_ANNUAL_INC"))
    .when(col("loan_amnt").isNull(), lit("NULL_LOAN_AMNT"))
    .when(col("loan_amnt") < 500, lit("LOW_LOAN_AMNT"))
    .when(col("loan_amnt") > 40000, lit("HIGH_LOAN_AMNT"))
    .when(~col("loan_status").isin(VALID_LOAN_STATUS), lit("INVALID_LOAN_STATUS"))
    .when(~col("grade").isin(VALID_GRADES), lit("INVALID_GRADE"))
    .when(col("dti").isNull(), lit("NULL_DTI"))
    .when(col("dti") < 0, lit("NEGATIVE_DTI"))
    .when(col("dti") > 100, lit("EXTREME_DTI"))
    .when(col("fico_range_low").isNull(), lit("NULL_FICO"))
    .when(col("fico_range_low") < 300, lit("LOW_FICO"))
    .when(col("fico_range_low") > 850, lit("HIGH_FICO"))
    .otherwise(lit("CLEAN"))
)

df_silver = df_validated.filter(col("rejection_reason") == "CLEAN").drop("rejection_reason").fillna({"emp_length": "Unknown"})
df_rejected = df_validated.filter(col("rejection_reason") != "CLEAN")

df_silver.write.format("delta").mode("overwrite").save(SILVER_PATH)
df_rejected.write.format("delta").mode("overwrite").save(REJECTED_PATH)

print(f"Silver records: {df_silver.count():,}")
print(f"Rejected records: {df_rejected.count():,}")
print(f"Duration: {datetime.now() - start_time}")

In [ ]:
df_rejected.groupBy("rejection_reason").count().orderBy("count", ascending=False).show(truncate=False)
df_silver.limit(10).show(truncate=False)